# ImageHRM ResNet ROCOv2 Reproduction

Mục tiêu: tái lập nhánh ResNet của paper `Hierarchical and Tiny Recursive Models for Medical Image Captioning` theo cấu trúc đơn giản: Load, EDA, Model, Eval, Test.

Notebook này tập trung vào ResNet18 frozen + ImageHRM H-M-L. Swin và FuseLIP chỉ dùng làm bối cảnh so sánh trong `core-features.md`.

## Load

Notebook nay dung mot flow duy nhat:

1. Tai bo file chinh thuc cua ROCOv2 tu Zenodo record `10821435` vao `data/rocov2_source/`.
2. Unzip `train_images.zip`, `valid_images.zip`, `test_images.zip` vao `data/rocov2/images/`.
3. Dung `train_captions.csv`, `valid_captions.csv`, `test_captions.csv` de tao `train.csv`, `val.csv`, `test.csv`.
4. Load `train.csv`, `val.csv`, `test.csv` va folder `images/`.

CSV chuan can hai cot:

- `image_path`: duong dan anh, tuong doi voi image root hoac absolute path.
- `caption`: caption/report text.

Phan EDA ben duoi van giu random sample de kiem tra nhanh anh va caption truoc khi train.

In [1]:
import zipfile
from pathlib import Path
import math
import random
import re

import pandas as pd
import requests
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

def find_project_root(start_path):
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "RBL1" / "imagehrm_resnet_reproduce").exists():
            return candidate
    raise FileNotFoundError("Khong tim thay project root chua pyproject.toml va RBL1/imagehrm_resnet_reproduce")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_SOURCE_ROOT = PROJECT_ROOT / "data" / "rocov2_source"
DATA_ROOT = PROJECT_ROOT / "data" / "rocov2"
IMAGE_ROOT = DATA_ROOT / "images"
TRAIN_CSV = DATA_ROOT / "train.csv"
VAL_CSV = DATA_ROOT / "val.csv"
TEST_CSV = DATA_ROOT / "test.csv"
DATA_ZIP_PATH = PROJECT_ROOT / "data" / "rocov2_full_for_kaggle.zip"
ROCO_RECORD_ID = "10821435"
ROCO_FILE_URLS = {
    "train_images.zip": f"https://zenodo.org/api/records/{ROCO_RECORD_ID}/files/train_images.zip/content",
    "valid_images.zip": f"https://zenodo.org/api/records/{ROCO_RECORD_ID}/files/valid_images.zip/content",
    "test_images.zip": f"https://zenodo.org/api/records/{ROCO_RECORD_ID}/files/test_images.zip/content",
    "train_captions.csv": f"https://zenodo.org/api/records/{ROCO_RECORD_ID}/files/train_captions.csv/content",
    "valid_captions.csv": f"https://zenodo.org/api/records/{ROCO_RECORD_ID}/files/valid_captions.csv/content",
    "test_captions.csv": f"https://zenodo.org/api/records/{ROCO_RECORD_ID}/files/test_captions.csv/content",
}
DOWNLOAD_CHUNK_SIZE = 1024 * 1024
REFRESH_DATA_ZIP = False

IMAGE_SIZE = 224
MAX_LENGTH = 512
BPE_VOCAB_SIZE = 8000
EMBEDDING_DIM = 512
H_CYCLES = 1
M_CYCLES = 1
L_CYCLES = 1
BATCH_SIZE = 8
NUM_EPOCHS = 50
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
USE_PRETRAINED_RESNET = True
EPOCHS_TO_RUN = NUM_EPOCHS

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", DEVICE)

device: mps


In [2]:
def normalize_caption(text):
    text = str(text).encode("ascii", errors="ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def download_file(url, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    print("downloading:", destination.name)
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with destination.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=DOWNLOAD_CHUNK_SIZE):
                if chunk:
                    handle.write(chunk)
    return destination


def ensure_roco_source_files(download_root, file_urls):
    download_root.mkdir(parents=True, exist_ok=True)
    local_paths = {}
    for filename, url in file_urls.items():
        local_path = download_root / filename
        if not local_path.exists():
            download_file(url, local_path)
        local_paths[filename] = local_path
    return local_paths


def ensure_unzipped_images(image_root, source_paths):
    required_paths = [image_root / "ROCOv2_2023_train_000001.jpg", image_root / "ROCOv2_2023_valid_000001.jpg", image_root / "ROCOv2_2023_test_000001.jpg"]
    if all(path.exists() for path in required_paths):
        return
    image_root.mkdir(parents=True, exist_ok=True)
    for zip_name in ["train_images.zip", "valid_images.zip", "test_images.zip"]:
        print("unzipping:", zip_name)
        with zipfile.ZipFile(source_paths[zip_name], "r") as archive:
            archive.extractall(image_root)


def build_split_csv(captions_path, output_csv_path, image_lookup):
    captions_table = pd.read_csv(captions_path)
    if "ID" not in captions_table.columns or "Caption" not in captions_table.columns:
        raise ValueError(f"Unexpected caption schema in {captions_path}: {captions_table.columns.tolist()}")
    split_table = captions_table[["ID", "Caption"]].rename(columns={"Caption": "caption"}).copy()
    split_table["image_path"] = split_table["ID"].map(image_lookup)
    missing_images = split_table[split_table["image_path"].isna()]["ID"].tolist()
    if missing_images:
        raise FileNotFoundError(f"Khong tim thay image cho {len(missing_images)} id dau tien: {missing_images[:5]}")
    split_table = split_table[["image_path", "caption"]]
    split_table.to_csv(output_csv_path, index=False)
    return output_csv_path


def load_caption_table(csv_path, image_root):
    table = pd.read_csv(csv_path)
    missing_columns = {"image_path", "caption"} - set(table.columns)
    if missing_columns:
        raise ValueError(f"missing columns in {csv_path}: {sorted(missing_columns)}")
    table = table[["image_path", "caption"]].copy()
    table["caption"] = table["caption"].map(normalize_caption)

    def resolve_image_path(value):
        path = Path(str(value))
        if path.is_absolute():
            return str(path)
        return str((image_root / path).resolve())

    table["image_path"] = table["image_path"].map(resolve_image_path)
    return table


In [3]:
roco_source_paths = ensure_roco_source_files(DATA_SOURCE_ROOT, ROCO_FILE_URLS)
print("source files ready:", len(roco_source_paths))


downloading: train_images.zip


ConnectionError: HTTPSConnectionPool(host='zenodo.org', port=443): Read timed out.

In [ ]:
ensure_unzipped_images(IMAGE_ROOT, roco_source_paths)

image_lookup = {image_path.stem: image_path.name for image_path in IMAGE_ROOT.iterdir() if image_path.is_file()}
DATA_ROOT.mkdir(parents=True, exist_ok=True)
build_split_csv(roco_source_paths["train_captions.csv"], TRAIN_CSV, image_lookup)
build_split_csv(roco_source_paths["valid_captions.csv"], VAL_CSV, image_lookup)
build_split_csv(roco_source_paths["test_captions.csv"], TEST_CSV, image_lookup)

train_table = load_caption_table(TRAIN_CSV, IMAGE_ROOT)
val_table = load_caption_table(VAL_CSV, IMAGE_ROOT)
test_table = load_caption_table(TEST_CSV, IMAGE_ROOT)

epochs_to_run = EPOCHS_TO_RUN

print("source root:", DATA_SOURCE_ROOT)
print("train csv:", TRAIN_CSV)
print("val csv:", VAL_CSV)
print("test csv:", TEST_CSV)
print("image root:", IMAGE_ROOT)
print("epochs to run:", epochs_to_run)
print("train rows:", len(train_table))
print("val rows:", len(val_table))
print("test rows:", len(test_table))
train_table.head()


In [ ]:
def refresh_data_zip(data_root, zip_path):
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    if zip_path.exists():
        zip_path.unlink()
    print("creating data zip:", zip_path)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for csv_name in ["train.csv", "val.csv", "test.csv"]:
            archive.write(data_root / csv_name, arcname=csv_name)
        for image_path in sorted((data_root / "images").iterdir()):
            if image_path.is_file():
                archive.write(image_path, arcname=f"images/{image_path.name}")
    size_gb = zip_path.stat().st_size / (1024 ** 3)
    print(f"data zip size: {size_gb:.2f} GiB")
    return zip_path


active_data_zip = DATA_ZIP_PATH
if REFRESH_DATA_ZIP:
    active_data_zip = refresh_data_zip(DATA_ROOT, DATA_ZIP_PATH)
print("active data zip:", active_data_zip)


## EDA

Kiểm tra nhanh số lượng, độ dài caption và ảnh mẫu. Nếu ảnh/caption không đúng ở đây thì chưa nên train model.

In [ ]:
def caption_token_count(text):
    return len(str(text).split())

for split_name, table in [("train", train_table), ("val", val_table), ("test", test_table)]:
    lengths = table["caption"].map(caption_token_count)
    print(split_name, {
        "rows": len(table),
        "min_tokens": int(lengths.min()),
        "mean_tokens": round(float(lengths.mean()), 2),
        "max_tokens": int(lengths.max()),
    })

In [ ]:
def show_samples(table, sample_count=3):
    sample_count = min(sample_count, len(table))
    fig, axes = plt.subplots(1, sample_count, figsize=(4 * sample_count, 4))
    if sample_count == 1:
        axes = [axes]
    for axis, (_, row) in zip(axes, table.sample(sample_count, random_state=SEED).iterrows()):
        image = Image.open(row["image_path"]).convert("RGB")
        axis.imshow(image)
        axis.set_title(row["caption"][:60], fontsize=9)
        axis.axis("off")
    plt.tight_layout()

show_samples(train_table)

## Model

Paper contract cho nhánh ResNet:

- ResNet18 dùng ImageNet weights, bỏ classification head, frozen.
- Visual feature 512 chiều được project sang embedding space.
- Caption tokens dùng ASCII/BPE, sequence length 512.
- Visual embedding được cộng vào từng token embedding: `Xt = Et + Ev`.
- ImageHRM triple-loop dùng `H -> M -> L` để mô phỏng planning, finding cluster và token execution.

In [ ]:
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, processors, trainers

SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]


def build_bpe_tokenizer(captions, vocab_size):
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL_TOKENS)
    tokenizer.train_from_iterator(captions, trainer=trainer)
    tokenizer.post_processor = processors.TemplateProcessing(
        single="<bos> $A <eos>",
        special_tokens=[("<bos>", tokenizer.token_to_id("<bos>")), ("<eos>", tokenizer.token_to_id("<eos>"))],
    )
    tokenizer.decoder = decoders.ByteLevel()
    return tokenizer


tokenizer = build_bpe_tokenizer(train_table["caption"].tolist(), BPE_VOCAB_SIZE)
PAD_ID = tokenizer.token_to_id("<pad>")
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")
VOCAB_SIZE = tokenizer.get_vocab_size()
print("vocab size:", VOCAB_SIZE)
print("pad/bos/eos:", PAD_ID, BOS_ID, EOS_ID)

In [ ]:
def encode_caption(caption, max_length=MAX_LENGTH):
    token_ids = tokenizer.encode(caption).ids[:max_length]
    token_ids = token_ids + [PAD_ID] * (max_length - len(token_ids))
    return torch.tensor(token_ids, dtype=torch.long)


def decode_tokens(token_ids):
    clean_ids = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id in {PAD_ID, BOS_ID}:
            continue
        if token_id == EOS_ID:
            break
        clean_ids.append(token_id)
    return tokenizer.decode(clean_ids).strip()


image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=ResNet18_Weights.DEFAULT.transforms().mean, std=ResNet18_Weights.DEFAULT.transforms().std),
])


class RocoCaptionDataset(Dataset):
    def __init__(self, table, transform):
        self.table = table.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        row = self.table.iloc[index]
        image = Image.open(row["image_path"]).convert("RGB")
        image_tensor = self.transform(image)
        token_ids = encode_caption(row["caption"])
        return image_tensor, token_ids


train_dataset = RocoCaptionDataset(train_table, image_transform)
val_dataset = RocoCaptionDataset(val_table, image_transform)
test_dataset = RocoCaptionDataset(test_table, image_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
class ResNet18Encoder(nn.Module):
    def __init__(self, pretrained=True, freeze=True):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        model = resnet18(weights=weights)
        self.features = nn.Sequential(*list(model.children())[:-1])
        if freeze:
            for parameter in self.features.parameters():
                parameter.requires_grad = False

    def forward(self, images):
        features = self.features(images)
        return features.flatten(1)


class ResNetLSTMBaseline(nn.Module):
    def __init__(self, vocab_size, embedding_dim=EMBEDDING_DIM):
        super().__init__()
        self.encoder = ResNet18Encoder(pretrained=USE_PRETRAINED_RESNET, freeze=True)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.image_to_hidden = nn.Linear(512, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, embedding_dim, batch_first=True)
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, images, token_ids):
        image_features = self.encoder(images)
        hidden_state = torch.tanh(self.image_to_hidden(image_features)).unsqueeze(0)
        cell_state = torch.zeros_like(hidden_state)
        token_embeddings = self.embedding(token_ids)
        sequence_output, _ = self.lstm(token_embeddings, (hidden_state, cell_state))
        return self.output(sequence_output)


class RecurrentBlock(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.norm = nn.LayerNorm(embedding_dim)
        self.feed_forward = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            nn.GELU(),
            nn.Linear(embedding_dim, embedding_dim),
        )

    def forward(self, state, update):
        return state + self.feed_forward(self.norm(state + update))


class ImageHRMResNetCaptioner(nn.Module):
    def __init__(self, vocab_size, embedding_dim=EMBEDDING_DIM, h_cycles=1, m_cycles=1, l_cycles=1):
        super().__init__()
        self.h_cycles = h_cycles
        self.m_cycles = m_cycles
        self.l_cycles = l_cycles
        self.encoder = ResNet18Encoder(pretrained=USE_PRETRAINED_RESNET, freeze=True)
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.visual_projection = nn.Linear(512, embedding_dim)
        self.low_block = RecurrentBlock(embedding_dim)
        self.middle_block = RecurrentBlock(embedding_dim)
        self.high_block = RecurrentBlock(embedding_dim)
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, images, token_ids):
        visual_features = self.encoder(images)
        visual_embedding = self.visual_projection(visual_features).unsqueeze(1)
        token_embedding = self.token_embedding(token_ids)
        merged_input = token_embedding + visual_embedding

        z_low = torch.zeros_like(merged_input)
        z_middle = torch.zeros_like(merged_input)
        z_high = torch.zeros_like(merged_input)

        for _ in range(self.h_cycles):
            middle_steps = max(1, self.m_cycles)
            for _ in range(middle_steps):
                for _ in range(self.l_cycles):
                    z_low = self.low_block(z_low, merged_input + z_middle)
                if self.m_cycles > 0:
                    z_middle = self.middle_block(z_middle, z_low + z_high)
            high_update = z_middle if self.m_cycles > 0 else z_low
            z_high = self.high_block(z_high, high_update)

        reasoning_state = merged_input + z_low + z_middle + z_high
        return self.output(reasoning_state)


model_variants = {
    "resnet_lstm": ResNetLSTMBaseline(VOCAB_SIZE).to(DEVICE),
    "imagehrm_dual": ImageHRMResNetCaptioner(VOCAB_SIZE, h_cycles=1, m_cycles=0, l_cycles=1).to(DEVICE),
    "imagehrm_triple": ImageHRMResNetCaptioner(VOCAB_SIZE, h_cycles=H_CYCLES, m_cycles=M_CYCLES, l_cycles=L_CYCLES).to(DEVICE),
}
triple_hrm_model = model_variants["imagehrm_triple"]

batch_images, batch_tokens = next(iter(train_loader))
with torch.no_grad():
    batch_logits = triple_hrm_model(batch_images.to(DEVICE), batch_tokens.to(DEVICE))
print("logits shape:", tuple(batch_logits.shape))
assert batch_logits.shape[:2] == batch_tokens.shape
assert batch_logits.shape[-1] == VOCAB_SIZE

## Eval

Train bang teacher forcing. Loss bo qua `<pad>` va predict token ke tiep. Notebook nay mac dinh chay tren bo du lieu da unzip tu `data/rocov2_full_for_kaggle.zip` voi `EPOCHS_TO_RUN = NUM_EPOCHS = 50`.

In [ ]:
def caption_loss(logits, token_ids):
    shifted_logits = logits[:, :-1, :].contiguous()
    shifted_targets = token_ids[:, 1:].contiguous()
    return F.cross_entropy(
        shifted_logits.view(-1, shifted_logits.size(-1)),
        shifted_targets.view(-1),
        ignore_index=PAD_ID,
    )


def train_one_epoch(model, data_loader, optimizer):
    model.train()
    total_loss = 0.0
    total_items = 0
    for images, token_ids in data_loader:
        images = images.to(DEVICE)
        token_ids = token_ids.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images, token_ids)
        loss = caption_loss(logits, token_ids)
        loss.backward()
        optimizer.step()
        batch_size = images.size(0)
        total_loss += float(loss.detach()) * batch_size
        total_items += batch_size
    return total_loss / max(total_items, 1)


@torch.no_grad()
def evaluate_loss(model, data_loader):
    model.eval()
    total_loss = 0.0
    total_items = 0
    for images, token_ids in data_loader:
        images = images.to(DEVICE)
        token_ids = token_ids.to(DEVICE)
        logits = model(images, token_ids)
        loss = caption_loss(logits, token_ids)
        batch_size = images.size(0)
        total_loss += float(loss) * batch_size
        total_items += batch_size
    return total_loss / max(total_items, 1)


def fit_model(model, train_loader, val_loader, epochs):
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    history = []
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_loss = evaluate_loss(model, val_loader)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"epoch {epoch:03d} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")
    return pd.DataFrame(history)

In [ ]:
def lcs_length(first_tokens, second_tokens):
    table = [[0] * (len(second_tokens) + 1) for _ in range(len(first_tokens) + 1)]
    for i, first_token in enumerate(first_tokens, start=1):
        for j, second_token in enumerate(second_tokens, start=1):
            if first_token == second_token:
                table[i][j] = table[i - 1][j - 1] + 1
            else:
                table[i][j] = max(table[i - 1][j], table[i][j - 1])
    return table[-1][-1]


def rouge_l_score(prediction, reference):
    prediction_tokens = prediction.split()
    reference_tokens = reference.split()
    if not prediction_tokens or not reference_tokens:
        return 0.0
    lcs = lcs_length(prediction_tokens, reference_tokens)
    precision = lcs / len(prediction_tokens)
    recall = lcs / len(reference_tokens)
    if precision + recall == 0:
        return 0.0
    return (2 * precision * recall) / (precision + recall)


@torch.no_grad()
def greedy_decode(model, image_tensor, max_new_tokens=80):
    model.eval()
    token_ids = [BOS_ID]
    image_tensor = image_tensor.unsqueeze(0).to(DEVICE)
    for _ in range(max_new_tokens):
        token_tensor = torch.tensor([token_ids], dtype=torch.long, device=DEVICE)
        logits = model(image_tensor, token_tensor)
        next_token_id = int(logits[0, -1].argmax())
        if next_token_id == EOS_ID:
            break
        token_ids.append(next_token_id)
    return decode_tokens(token_ids)


@torch.no_grad()
def evaluate_caption_samples(model, dataset, sample_count=5):
    rows = []
    for index in range(min(sample_count, len(dataset))):
        image_tensor, token_ids = dataset[index]
        prediction = greedy_decode(model, image_tensor)
        reference = decode_tokens(token_ids.tolist())
        rows.append({
            "index": index,
            "prediction": prediction,
            "reference": reference,
            "rouge_l": rouge_l_score(prediction, reference),
        })
    return pd.DataFrame(rows)

In [ ]:
history = fit_model(triple_hrm_model, train_loader, val_loader, epochs=epochs_to_run)
history

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history["epoch"], history["train_loss"], label="train_loss")
plt.plot(history["epoch"], history["val_loss"], label="val_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.tight_layout()

## Test

Phần này kiểm tra held-out samples và gom bảng metric tối thiểu. ROUGE-L bên dưới là implementation nội bộ để kiểm tra nhanh. Để report CIDEr đúng paper, nên dùng cùng package/config với tác giả; paper không công bố chi tiết này.

In [ ]:
test_predictions = evaluate_caption_samples(triple_hrm_model, test_dataset, sample_count=5)
test_predictions

In [ ]:
reported_resnet_rows = pd.DataFrame([
    {"model": "ResNet+LSTM", "backbone": "ResNet18", "h_m_l": "N/A", "paper_rouge_l": 0.106, "paper_cider": 0.310},
    {"model": "ImageHRM Dual", "backbone": "ResNet18", "h_m_l": "1/0/1", "paper_rouge_l": 0.125, "paper_cider": 0.420},
    {"model": "ImageHRM Triple", "backbone": "ResNet18", "h_m_l": "1/1/1", "paper_rouge_l": 0.157, "paper_cider": 0.478},
])

current_summary = pd.DataFrame([
    {
        "model": "ImageHRM Triple",
        "backbone": "ResNet18",
        "h_m_l": f"{H_CYCLES}/{M_CYCLES}/{L_CYCLES}",
        "epochs_run": epochs_to_run,
        "val_loss": float(history.iloc[-1]["val_loss"]),
        "sample_rouge_l": float(test_predictions["rouge_l"].mean()) if len(test_predictions) else math.nan,
        "paper_comparable": TRAIN_CSV.exists() and VAL_CSV.exists() and TEST_CSV.exists() and epochs_to_run == NUM_EPOCHS,
    }
])

print("Paper rows to reproduce:")
display(reported_resnet_rows)
print("Current run summary:")
display(current_summary)

## Unresolved Questions

- ROCOv2 source/layout cụ thể của tác giả là gì?
- Batch size, learning rate schedule và weight decay gốc là gì?
- BPE vocab size và training corpus chính xác là gì?
- ACT loss trong paper được tính chính xác ra sao?
- CIDEr được tính bằng package/config nào?
- Nên theo H/M/L `1/1/1` trong result table hay block depth `2/2/2` trong implementation notes?